# SVG Generation from Text Prompts

**DL Spring 2026 - SVG Generation**

This notebook generates SVG code from text prompts using a LoRA-finetuned Qwen2.5-Coder-3B-Instruct model.

## Approach
- **Base model:** Qwen/Qwen2.5-Coder-3B-Instruct
- **Fine-tuning:** LoRA (rank 16) on the competition training set
- **Training:** Supervised fine-tuning on NYU HPC (H100 GPUs)
- **Inference:** Greedy decoding with SVG extraction and validation

## AI Usage Disclosure
LLM-based coding assistants were used for implementation support and debugging. Final experiments, analysis, and submission decisions were reviewed by the authors.

## Resources
- Training code: public repository accompanying the report
- Base model: https://huggingface.co/Qwen/Qwen2.5-Coder-3B-Instruct
- Libraries: transformers, peft, torch


In [ ]:
import csv
import re
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

## Configuration

In [ ]:
# --- Paths ---
BASE_MODEL_DIR = "/kaggle/input/models/qwen-lm/qwen2.5-coder/transformers/3b-instruct/1"
ADAPTER_DIR = "/kaggle/input/datasets/adamfw/svg-lora-adapter/svg-lora-adapter"
TEST_CSV = "/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/test.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

# --- Generation settings ---
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup for 256x256 canvases. "
    "Return only SVG code with a single root <svg> element."
)
MAX_NEW_TOKENS = 2048
BATCH_SIZE = 16

# --- Constants ---
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)
BLANK_FALLBACK = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256"></svg>'

In [ ]:
def prompt_prefix(prompt: str, system_prompt: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{system_prompt}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def extract_svg(text: str) -> str:
    match = SVG_REGEX.search(text)
    return match.group(0).strip() if match else text.strip()


def validate_svg(svg_text: str, max_length: int = 16000, max_paths: int = 256) -> bool:
    """Validate SVG against Kaggle constraints."""
    raw = svg_text.strip()
    if not raw or not raw.lower().startswith("<svg"):
        return False
    if len(raw) > max_length:
        return False
    if 'viewBox="0 0 256 256"' not in raw:
        return False
    try:
        root = ET.fromstring(raw)
    except ET.ParseError:
        return False
    path_count = sum(1 for elem in root.iter() if elem.tag.split("}")[-1] == "path")
    if path_count > max_paths:
        return False
    return True


def generate_batch(model, tokenizer, prompts, system_prompt, max_new_tokens, device):
    """Generate SVGs for a batch of prompts."""
    batch_text = [prompt_prefix(p, system_prompt) for p in prompts]
    inputs = tokenizer(
        batch_text,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    results = []
    for gen in generated_ids:
        text = tokenizer.decode(gen, skip_special_tokens=False)
        svg = extract_svg(text).strip()
        if not validate_svg(svg):
            svg = BLANK_FALLBACK
        results.append(svg)
    return results

## Load Test Data

In [ ]:
test_rows = []
with open(TEST_CSV, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_rows.append(row)

prompts = [row["prompt"] for row in test_rows]
ids = [row["id"] for row in test_rows]
print(f"Loaded {len(test_rows)} test prompts")

## Load Model + LoRA Adapter

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"Device: {device}, dtype: {dtype}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    local_files_only=True,
    dtype=dtype,
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.to(device)
model.eval()
model.config.use_cache = True
print("Model loaded and ready")

## Generate SVGs

In [ ]:
all_svgs = []
total_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

t0 = time.time()
for i in range(0, len(prompts), BATCH_SIZE):
    batch_num = i // BATCH_SIZE + 1
    batch_prompts = prompts[i : i + BATCH_SIZE]
    batch_svgs = generate_batch(
        model, tokenizer, batch_prompts,
        SYSTEM_PROMPT, MAX_NEW_TOKENS, device,
    )
    all_svgs.extend(batch_svgs)

    valid_count = sum(1 for s in batch_svgs if s != BLANK_FALLBACK)
    elapsed = time.time() - t0
    if batch_num % 10 == 0 or batch_num == total_batches:
        print(f"  batch {batch_num}/{total_batches}  "
              f"generated={len(all_svgs)}/{len(prompts)}  "
              f"valid={valid_count}/{len(batch_svgs)}  "
              f"elapsed={elapsed:.0f}s")

elapsed = time.time() - t0
total_valid = sum(1 for s in all_svgs if s != BLANK_FALLBACK)
print(f"\nDone: {len(all_svgs)} rows, {total_valid} valid, "
      f"{len(all_svgs) - total_valid} fallback, {elapsed:.0f}s")

## Write submission.csv

In [ ]:
assert len(all_svgs) == len(test_rows), (
    f"Row count mismatch: {len(all_svgs)} vs {len(test_rows)}"
)

output_path = Path(OUTPUT_CSV)
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "svg"])
    for row_id, svg in zip(ids, all_svgs):
        writer.writerow([row_id, svg.replace("\n", " ").replace("\r", "")])

# Verify
with output_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ["id", "svg"], f"Bad header: {reader.fieldnames}"
    output_rows = list(reader)

assert len(output_rows) == len(test_rows), (
    f"Output count {len(output_rows)} != expected {len(test_rows)}"
)

print(f"submission.csv written: {len(output_rows)} rows, verified")